Here we will try some simple models like
1. Logistic Regression 
2. RandomForest

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GroupKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import confusion_matrix, accuracy_score

In [92]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [93]:
df = pd.read_csv("data/train/train_tabular.csv")
df.drop( columns=["flight_id", "failure_mode"], inplace = True)

In [94]:
target_column = "failure_within_horizon"
group_column = "drone_id"

In [95]:
X = df.drop(columns = [target_column, "drone_id"])
y = df[target_column]
g = df["drone_id"]

In [96]:
categorical_columns = ["model", "motor_type", "firmware_version", "operator_region"]
numerical_columns = X.select_dtypes(include="number").columns

In [97]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
        ("num", StandardScaler(), numerical_columns),
    ]
)

In [103]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [104]:
models = [
    RandomForestClassifier(n_estimators=100, random_state=42),
    LogisticRegression()
]

In [ ]:
for model in models:
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_val_score(
        pipeline, X, y, groups=g, cv = GroupKFold(n_splits=5), scoring="average_precision"
    )

    print(f"Model: {model}, Mean: {scores.mean()}, Std: {scores.std()}")

Model: RandomForestClassifier(random_state=42), Mean: 0.2916413442124017, Std: 0.030738195450363864
Model: LogisticRegression(), Mean: 0.32297585720716826, Std: 0.03914638117420622
